# Partial correlation at baseline (IEEE)

## Purpose

Compute partial correlation between metrics and UPDRS score at baseline.
Metrics are cortical thickness, area, volume and subcortical volume.

## Definition

Pingouin [method](https://pingouin-stats.org/build/html/generated/pingouin.partial_corr.html#pingouin.partial_corr)

Partial correlation [1] measures the degree of association between x and y, after removing the effect of one or more controlling variables (covar or $Z$). Practically, this is achieved by calculating the correlation coefficient between the residuals of two linear regressions:

$$x \sim Z, y \sim Z$$

Like the correlation coefficient, the partial correlation coefficient takes on a value in the range from –1 to 1, where 1 indicates a perfect positive association.

The semipartial correlation is similar to the partial correlation, with the exception that the set of controlling variables is only removed for either x or y, but not both.

Pingouin uses the method described in [2] to calculate the (semi)partial correlation coefficients and associated p-values. This method is based on the inverse covariance matrix and is significantly faster than the traditional regression-based method. Results have been tested against the ppcor R package.

## Get info about subjects

In [33]:
%load_ext cudf.pandas
import pandas as pd
import pingouin as pg
import os
from pathlib import Path

# suppress warnings
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*column_view.*")

output_dir = Path() / "partial_correlation_ieee"
output_dir.mkdir(parents=True, exist_ok=True)
print("Output directory:", output_dir.absolute())


The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas
Output directory: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/notebooks/partial_correlation_ieee


In [34]:
def get_clinical_baseline_pcorr():
    df_clinical = pd.read_csv("../pd_clinical.csv")
    print(f"Load cohort stats: {os.path.abspath('../pd_clinical.csv')}")
    columns = [
        "PATNO",
        "first_visit",
        "second_visit",
        "dx_group",
        "SEX",
        "AGE_AT_VISIT",
        "UPDRS",
    ]
    df_clinical["first_visit"] = (
        "sub-"
        + df_clinical["PATNO"].astype(str)
        + "_ses-"
        + df_clinical["EVENT_ID"].astype(str)
    )
    df_clinical["second_visit"] = (
        "sub-"
        + df_clinical["PATNO"].astype(str)
        + "_ses-"
        + df_clinical["NEXT_VISIT"].astype(str)
    )
    df_clinical = df_clinical[df_clinical.dx_group == "PD-non-MCI"]
    df_clinical.rename(columns={"NP3TOT": "UPDRS"}, inplace=True)
    print(f"Number of PD-non-MCI subjects: {df_clinical.shape[0]}")
    return df_clinical[columns]


df_clinical = get_clinical_baseline_pcorr()


Load cohort stats: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/pd_clinical.csv
Number of PD-non-MCI subjects: 119


## Partial correlation

In [35]:
def read_table(hemi, measure):
    df = pd.read_csv(f"table_ieee/{hemi}.aparc.{measure}.tsv", sep="\t")
    df["hemi"] = hemi
    df.columns = [c.replace(f"{hemi}.", "") for c in df.columns]
    df.columns = [c.replace(f"{hemi}_", "") for c in df.columns]
    df.columns = [c.replace(f"_{measure}", "") for c in df.columns]
    df.rename(columns={f"aparc.{measure}": "first_visit"}, inplace=True)
    return df


def read_measure(measure):
    lh = read_table("lh", measure)
    rh = read_table("rh", measure)
    return pd.concat([lh, rh], axis=0)


def get_baseline_pcorr(metric):
    df = read_measure(metric)
    df = df.melt(id_vars=["first_visit", "hemi"], var_name="region", value_name=metric)
    df = pd.merge(df, df_clinical, on="first_visit")
    df = df[
        ["first_visit", "hemi", "region", metric, "dx_group", "AGE_AT_VISIT", "SEX"]
    ]
    return df


### Cortical

In [36]:
def compute_partial_correlation(metric, clinical_df, force=False):
    baseline_df = get_baseline_pcorr(metric)
    baseline_df = pd.merge(
        baseline_df,
        clinical_df,
        left_on="first_visit",
        right_on="first_visit",
        suffixes=("", "_clinical"),
    )
    baseline_df = baseline_df[
        ["first_visit", "region", metric, "hemi", "AGE_AT_VISIT", "SEX", "UPDRS"]
    ]

    columns = ["region", "hemisphere", "r", "p-val", "n"]
    partial_correlation_df = pd.DataFrame(columns=columns)

    errors = []
    hemispheres = ["lh", "rh"]
    for hemi in hemispheres:
        for region in baseline_df["region"].unique():
            data = baseline_df[
                (baseline_df["region"] == region) & (baseline_df["hemi"] == hemi)
            ]
            pc = pg.partial_corr(
                data=data,
                x=metric,
                y="UPDRS",
                covar=["AGE_AT_VISIT", "SEX"],
                method="pearson",
            )

            (r, pval, n) = (pc["r"][0], pc["p-val"][0], pc["n"][0])
            idx = len(partial_correlation_df)
            partial_correlation_df.loc[idx] = [region, hemi, r, pval, n]

    for error in errors:
        print(error)

    filename = output_dir / f"partial_correlation_baseline_{metric}.csv"
    partial_correlation_df.to_csv(filename, index=False)

    return partial_correlation_df


In [37]:
pcorr_thickness = compute_partial_correlation("thickness", df_clinical)
pcorr_area = compute_partial_correlation("area", df_clinical)
pcorr_volume = compute_partial_correlation("volume", df_clinical)

In [38]:
pcorr_thickness[pcorr_thickness["p-val"] < 0.05]

,region,hemisphere,r,p-val,n
19,pericalcarine,lh,-0.261933,0.004334,119
20,postcentral,lh,-0.225874,0.014337,119
22,precentral,lh,-0.189915,0.040277,119
27,superiorparietal,lh,-0.186456,0.044128,119
28,superiortemporal,lh,-0.193154,0.036930,119
56,pericalcarine,rh,-0.253151,0.005890,119
57,postcentral,rh,-0.250994,0.006341,119
69,transversetemporal,rh,-0.221607,0.016339,119


In [39]:
pcorr_area[pcorr_area["p-val"] < 0.05]

,region,hemisphere,r,p-val,n
7,inferiortemporal,lh,-0.186012,0.044645,119
21,posteriorcingulate,lh,-0.236214,0.010347,119
43,inferiorparietal,rh,-0.194273,0.035829,119


In [40]:
pcorr_volume[pcorr_volume["p-val"] < 0.05]

,region,hemisphere,r,p-val,n
7,inferiortemporal,lh,-0.191017,0.039111,119
20,postcentral,lh,-0.203845,0.027493,119
21,posteriorcingulate,lh,-0.289465,0.001548,119
28,superiortemporal,lh,-0.186647,0.043908,119
32,transversetemporal,lh,-0.189495,0.040729,119
36,bankssts,rh,-0.189912,0.040281,119
42,inferiorparietal,rh,-0.224689,0.014871,119
69,insula,rh,-0.210244,0.022893,119


## Subcortical Volume


In [41]:
df = pd.read_csv("table_ieee/aseg.volume.tsv", sep="\t")
df.rename(columns={"Measure:volume": "first_visit"}, inplace=True)
df = df.melt(id_vars=["first_visit"], var_name="region", value_name="volume")
df = pd.merge(
    df,
    df_clinical,
    left_on="first_visit",
    right_on="first_visit",
    suffixes=("", "_clinical"),
)
df = df[["first_visit", "region", "volume", "dx_group", "AGE_AT_VISIT", "SEX", "UPDRS"]]

columns = ["region", "r", "p-val", "n"]
partial_correlation_df = pd.DataFrame(columns=columns)

for region in df["region"].unique():
    data = df[(df["region"] == region)]
    pc = pg.partial_corr(
        data=data,
        x="volume",
        y="UPDRS",
        covar=["AGE_AT_VISIT", "SEX"],
        method="pearson",
    )

    (r, pval, n) = (pc["r"][0], pc["p-val"][0], pc["n"][0])
    idx = len(partial_correlation_df)
    partial_correlation_df.loc[idx] = [region, r, pval, n]

filename = output_dir / "partial_correlation_baseline_subcortical_volume.csv"
partial_correlation_df.to_csv(filename, index=False)

partial_correlation_df[partial_correlation_df["p-val"] < 0.05]


,region,r,p-val,n
5,Left-Caudate,-0.192892,0.037192,119
6,Left-Putamen,-0.228057,0.013398,119
11,Left-Hippocampus,-0.189702,0.040506,119
12,Left-Amygdala,-0.185723,0.044983,119
24,Right-Putamen,-0.258844,0.004833,119
26,Right-Hippocampus,-0.220926,0.016680,119
27,Right-Amygdala,-0.195347,0.034798,119
